## Lecture Notes - Join Tables by Columns ##

**Helpful Resource:**
- [Python Reference](http://data8.org/sp22/python-reference.html): Cheat sheet of helpful array & table methods used in Data 8!

**Recommended Readings:**
- [Joining Tables by Columns](https://inferentialthinking.com/chapters/08/4/Joining_Tables_by_Columns.html)


In [ ]:
# import modules to be used in this notebook

from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings

In [ ]:
my_cones = Table().read_table("cones.csv")
my_cones

In [ ]:
# Create a ratings table with columns of Category, Star and Price Change Fraction 
#   based on the values in Price Change Fraction column
#   some prices will be increased and some will be reduced  

categories = make_array("strawberry", "chocolate", "bubblegum")
stars = make_array(2.5, 3.5, 4.2)
price_changes = make_array(0.95, 1.05, 1.03)

ratings = Table().with_columns(
        "Category", categories,
        "Star", stars,
        "Price Change Fraction", price_changes
)

ratings

In [ ]:
# combine the cones and ratings table togather using tbl.join and
#   assign the combined table view to variable new_cones_tbl

new_cones_tbl = my_cones.join("Flavor", ratings, "Category")
new_cones_tbl

In [ ]:
ratings.join("Category", my_cones, "Flavor")

In [ ]:
# create a function to compute the new prices for each flavor

def new_price(price, fraction):
    return price * fraction

# call the function to return an array of new prices and
#  assign the array to the variable updated_prices

updated_prices = new_price(new_cones_tbl.column("Price"), new_cones_tbl.column("Price Change Fraction"))
updated_prices

In [ ]:
# append the updated_prices array to new_cones_tbl view

cones_with_new_price = new_cones_tbl.with_columns("New Price", updated_prices)
cones_with_new_price

In [ ]:
# move the "Price" column next the "New Price" column
# step 1: append a new column "Old Price" with the data in "Price" column
#         to the table
# step 2: drop the "Price" column

# step 1 here
arranged_col = cones_with_new_price.with_columns("Old Price", new_cones_tbl.column("Price"))

# step 2 here
arranged_col.drop("Price")

In [ ]:
# we could also use tbl.select() to rearrange the columns

rearranged_col = cones_with_new_price.select("Flavor", "Color", "Price", "New Price", "Star").relabeled("Price", "Old Price")
rearranged_col

In [ ]:
# some fancy join showing how to look up common records between two tables
# here we are interested in the same cars that were made in 2015 as well as in 2025

#cars2025 = Table().read_table('Cars2025.csv')
#cars2015 = Table().read_table('Cars2015_v1.csv')
#cars_after_join = cars2025.join(['Make', 'Model'], cars2015, ['Make', 'Model'])
#cars_after_join

In [ ]:
cars2025 = Table().read_table('Cars2025.csv')
cars2025

In [ ]:
cars2015 = Table().read_table('Cars2015_v1.csv')
cars2015

In [ ]:
cars_after_join = cars2025.join(['Make', 'Model'], cars2015, ['Make', 'Model'])
cars_after_join

In [ ]:
# an example of using Table.apply on multiple columns simultaneously
def price_change_percent(old_price, new_price):
    diff = new_price - old_price
    change_percent = round(diff / old_price * 100)
    return str(change_percent) + '%'

low_change = cars_after_join.apply(price_change_percent, 'LowPrice_2', 'LowPrice')
high_change = cars_after_join.apply(price_change_percent, 'HighPrice_2', 'HighPrice')

cars_after_join.select('Make', 'Model').with_columns('LowPrice Change', low_change, 'HighPrice Change', high_change)
